# Uni-D 후보 모델 실험

이 노트북은 실험 설계와 확인용입니다. **학습과 추론은 여기서 실행하지 않습니다.** 오래 걸리는 학습은 VS Code 터미널의 tmux 안에서 `python train.py`로 돌립니다. 연결이 끊기거나 노트북 덮개를 닫아도 그때의 학습은 유지됩니다.

코드의 기준은 `model.py`입니다. `train.py`와 `test.py`가 그 함수로 학습하고 예측합니다. 운영진은 `train.py`를 실행하지 않고 아래 한 줄만 돌립니다.

```bash
python test.py --input <CSV> --image_root <이미지 폴더> --output <저장 경로>
```

선택 기준은 Train 실험 단위 교차검증 Macro F1입니다. Validation 점수는 후보를 고른 뒤에 한 번만 확인하고, 그 숫자로 모델을 고르지 않습니다. Test는 추론에만 씁니다.

## 후보

| 순위 | 모델 | 종류 | 입력 |
|---|---|---|---|
| 1 | LightGBM, XGBoost, CatBoost, Extra Trees, HistGradientBoosting | 표 | 확장 피처 |
| 1 | cumulative | 표 | “k보다 큰가” 이항 5개 |
| 2 | EfficientNet-B0, ResNet18, MobileNetV3-Small | 딥러닝 | 현재 열화상 + 센서 피처 |
| 2 | smallcnn | 딥러닝 | 열화상 전용 작은 합성곱망 + 센서 피처 |
| 2 | gru_efficientnet | 딥러닝 | 현재 열화상 + 과거 300초 센서 |
| 3 | ensemble | 조합 | 표 모델 여섯 개의 확률 평균 |
| 3 | `--head coral` | 조합 | 딥러닝 출력을 순서형 5개로 |

확장 피처는 센서 5개, 제공 CSV에 있는 가스·`thermal_t`·`atmp_t` 등, 열화상 밝기·고온 영역, 30·60·120·300초의 과거 평균·최댓값·변화·온도 기울기, 실험 시작 시각 대비 변화입니다. `indu_t.1`은 넣지 않습니다.

확률은 평균하고, 확정 라벨의 다수결은 쓰지 않습니다. 조합 비율은 학습 스크립트 안에서 Train 교차검증으로만 정해지며 기본 앙상블은 같은 비중입니다.

## 공통 규칙

| 항목 | 방법 |
|---|---|
| 분할 | Train 실험을 GroupKFold. 같은 실험이 학습과 검증에 동시에 들어가지 않음 |
| 표 모델 | 5분할 Macro F1. 4단계·5단계 F1을 같이 기록 |
| 딥러닝 | 기본 1개 홀드아웃으로 epoch을 고른 뒤 Train 전체로 다시 학습 |
| Validation | 선택 후 한 번만 확인 |
| 시계열 | 같은 실험, 현재 시각 이전과 현재. 단위는 초 |
| 전처리 | 중앙값·표준화는 그 모델을 학습하는 행에서만 계산해 가중치에 저장 |
| 추론 | `eval`, 고정 시드, 고정 리사이즈. 무작위 뒤집기는 학습에만 |
| 이미지 | 샘플마다 계산. 미리 만들어 둔 피처 파일을 `test.py`가 읽지 않음 |

사전학습 가중치는 대회 시작 전에 공개된 torchvision ImageNet-1K V1입니다. 이 대회 데이터로 학습된 외부 가중치와 외부 추론 API는 쓰지 않습니다.

In [ ]:
import model as M

print('표 모델:', ', '.join(M.TABULAR_MODELS))
print('딥러닝:', ', '.join(M.DEEP_MODELS))
print('조합:', ', '.join(M.COMBINATION_MODELS))
print('윈도우(초):', M.WINDOWS)
print('시드:', M.SEED)
print('이 셀은 모델을 학습하지 않습니다. 학습은 tmux의 train.py에서만 실행합니다.')


## tmux에서 실행할 명령

세션을 만든 뒤 화면이 바뀌면 나머지를 입력합니다. 로그는 `train.log`에 남고, 끝나면 교차검증 Macro F1과 Validation Macro F1이 출력됩니다.

```bash
tmux new -s train-<내이름>
source ~/venv/bin/activate
cd ~/<내이름>/Uni-Dthon-Team-16
pip install lightgbm xgboost catboost
D=~/data/battery_data_light
python -u train.py --data_dir $D --compare tabular --model_out weights/model.joblib 2>&1 | tee train.log
```

`Ctrl+B` 다음 `D`로 세션에서 나옵니다. 다시 들어가려면 `tmux attach -t train-<내이름>`. 세션 목록은 `tmux ls`입니다.

남아 있는 딥러닝은 smallcnn, mobilenet, resnet18입니다. 한 번에 하나만 실행합니다. gru_efficientnet은 공용 서버 CPU를 오래 붙잡으므로 이 순서에 넣지 않습니다. 이미지 피처와 torch 스레드는 기본 2개로 제한됩니다. 딥러닝을 돌릴 때는 다른 터미널에서 `watch -n 2 nvidia-smi`로 GPU를 확인합니다. 끝낼 때는 `Ctrl+C`입니다. 표 모델만 돌릴 때 GPU 사용률 0%는 정상입니다.

```bash
python -u train.py --data_dir $D --model efficientnet --model_out weights/efficientnet.joblib 2>&1 | tee train_efficientnet.log
python -u train.py --data_dir $D --model smallcnn --model_out weights/smallcnn.joblib 2>&1 | tee train_smallcnn.log
python -u train.py --data_dir $D --model mobilenet --model_out weights/mobilenet.joblib 2>&1 | tee train_mobilenet.log
python -u train.py --data_dir $D --model resnet18 --model_out weights/resnet18.joblib 2>&1 | tee train_resnet.log
```

각 후보는 `weights/<모델>.joblib`에 남습니다. 리더보드에 낸 가중치는 지우지 않습니다. 최종으로 고른 파일만 `weights/model.joblib`로 두면 `test.py` 기본 경로와 같습니다.

```bash
python test.py --input $D/test/test.csv --image_root $D/test --output submission.csv
```

제출 전에 `check_submission.py`로 재현, 같은 입력 재실행, 행 순서, 실험 분리, 미래 행 제거가 모두 통과하는지 확인합니다. 최종 파일은 이 `test.py`가 만든 CSV입니다.

## 로그에서 볼 표

`train.py`가 아래 열로 한 줄씩 출력합니다. `cv_f1`이 가장 높은 모델을 저장하고, `val_f1`은 참고값입니다.

| 열 | 의미 |
|---|---|
| cv_f1 | Train 실험 교차검증 Macro F1. 선택 기준 |
| cv_std | 분할 간 표준편차. 딥러닝 1분할이면 `-` |
| f1_4, f1_5 | 그 교차검증의 4단계·5단계 F1 |
| val_f1 | 학습에 넣지 않은 Validation Macro F1 |

같은 평균이면 목록에서 앞에 있는 모델을 남깁니다. 4·5단계 F1이 무너진 후보는 평균이 조금 높아도 최종 파일로 올리지 않습니다.